In [1]:
import os

import pandas as pd
from dotenv import load_dotenv
from google import genai

In [2]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY não encontrada no arquivo .env")

client = genai.Client(api_key=api_key)
MODEL = "gemini-2.5-flash"

In [3]:
df = pd.read_csv("../data/raw/dataset.csv")
df.head()

,ID,Text,FT,FP,FR,FS,GP1,GP2,GP3,GP4,GP5,GP6,GP7,Type,Course,Intructor
0,20,Ter cuidado com as cópias fieis da internet. P...,0,1,0,0,0,0,0,0,0,0,0,Individual,literature,1
1,22,"Texto bem escrito, apenas rever as margens (co...",0,1,0,0,0,0,0,0,0,1,0,Individual,literature,1
2,23,Ler outros textos sobre o assunto para lher da...,1,0,0,0,0,0,0,0,0,1,0,Individual,literature,1
3,24,"Procure ler mais, para escrever mais acerca do...",1,1,0,0,0,0,0,0,0,1,0,Individual,literature,1
4,25,"Não realizar cópia da internet na íntegra, pro...",0,1,0,0,0,0,0,0,0,0,0,Individual,literature,1


In [4]:
texto = df["Text"].iloc[0]
print(texto)

Ter cuidado com as cópias fieis da internet. Procurar ler e escrever com suas palavras a partir do que entendeu.


In [5]:
prompt = f"""
Você é um especialista em avaliação educacional.

Classifique o feedback nas categorias abaixo. Para cada uma, responda 1 (presente) ou 0 (ausente).

GP1: Esclarece o que é um bom desempenho.
GP2: Promove autoavaliação e reflexão do aluno.
GP3: Fornece informações de qualidade sobre o aprendizado.
GP4: Incentiva diálogo entre aluno e professor/colegas.
GP5: Incentiva motivação e autoestima.
GP6: Ajuda a reduzir a diferença entre desempenho atual e desejado.
GP7: Fornece informações úteis para o professor ajustar o ensino.

Responda no formato:
GP1: 0 ou 1
GP2: 0 ou 1
GP3: 0 ou 1
GP4: 0 ou 1
GP5: 0 ou 1
GP6: 0 ou 1
GP7: 0 ou 1

Feedback:
{texto}
"""

In [6]:
response = client.models.generate_content(model=MODEL, contents=prompt)
print(response.text)

GP1: 1
GP2: 1
GP3: 1
GP4: 0
GP5: 0
GP6: 1
GP7: 1


## Processamento em lote (Batch Mode)

O Gemini também tem uma API de lote (Batch Mode), que aceita uma lista de requisições e processa tudo de forma assíncrona.

Vantagens:
- **50% mais barato** que chamadas individuais.
- Prazo de processamento: normalmente rápido, com até 24h no pior caso.
- Ótimo para o nosso caso: 1000 feedbacks no dataset — nada de ficar chamando a API um por um.

O fluxo é: (1) montar a lista de requisições inline, cada uma com um `metadata` contendo um `custom_id` próprio (a API ecoa esse metadata de volta na resposta, então não precisamos confiar na ordem); (2) criar o job de lote; (3) consultar o status até terminar; (4) ler as respostas casando pelo `custom_id`.

**Abaixo, o lote roda com o dataset completo (1000 feedbacks).** O tempo de processamento não é garantido — pode levar de poucos minutos até (no pior caso) 24h. Se quiser testar mais rápido antes de rodar tudo, troque `linhas = df` por `linhas = df.head(5)`.

In [ ]:
def montar_prompt(texto: str) -> str:
    return f"""
Você é um especialista em avaliação educacional.

Classifique o feedback nas categorias abaixo. Para cada uma, responda 1 (presente) ou 0 (ausente).

GP1: Esclarece o que é um bom desempenho.
GP2: Promove autoavaliação e reflexão do aluno.
GP3: Fornece informações de qualidade sobre o aprendizado.
GP4: Incentiva diálogo entre aluno e professor/colegas.
GP5: Incentiva motivação e autoestima.
GP6: Ajuda a reduzir a diferença entre desempenho atual e desejado.
GP7: Fornece informações úteis para o professor ajustar o ensino.

Responda no formato:
GP1: 0 ou 1
GP2: 0 ou 1
GP3: 0 ou 1
GP4: 0 ou 1
GP5: 0 ou 1
GP6: 0 ou 1
GP7: 0 ou 1

Feedback:
{texto}
"""

In [ ]:
import time

linhas = df  # dataset completo (1000 feedbacks). Para testar rápido, use: df.head(5)

inline_requests = [
    {
        "contents": [{"role": "user", "parts": [{"text": montar_prompt(row.Text)}]}],
        "metadata": {"custom_id": f"feedback-{row.ID}"},
    }
    for row in linhas.itertuples()
]

batch_job = client.batches.create(
    model=MODEL,
    src=inline_requests,
    config={"display_name": "feedback-formativo-lote"},
)
print(f"Lote criado: {batch_job.name} ({len(inline_requests)} requisições)")

In [ ]:
estados_finais = {
    "JOB_STATE_SUCCEEDED",
    "JOB_STATE_FAILED",
    "JOB_STATE_CANCELLED",
    "JOB_STATE_EXPIRED",
}

batch_job = client.batches.get(name=batch_job.name)
while batch_job.state.name not in estados_finais:
    print(f"Status atual: {batch_job.state.name}")
    time.sleep(15)
    batch_job = client.batches.get(name=batch_job.name)

print(f"Lote concluído com status: {batch_job.state.name}")

In [ ]:
# Casa cada resposta com seu feedback original pelo custom_id no metadata
resultados = {}

if batch_job.state.name == "JOB_STATE_SUCCEEDED":
    for resposta in batch_job.dest.inlined_responses:
        custom_id = resposta.metadata["custom_id"]
        if resposta.response:
            resultados[custom_id] = resposta.response.text
        else:
            resultados[custom_id] = f"[erro: {resposta.error}]"

    for custom_id in sorted(resultados):
        print(f"--- {custom_id} ---")
        print(resultados[custom_id])
        print()
else:
    print(f"Lote não teve sucesso: {batch_job.error}")